# D4: continuous batching and recomputation preemption

D3 answered: **which physical KV blocks belong to a request?** D4 adds a scheduler to answer: **which requests should process tokens in the next model forward?** A request is one independent token sequence. A forward processes a selected group of token rows through the model.

We will first let a new request join while older requests are still running. Then we will deliberately use a small KV pool, pause a newer request, release its blocks, and rebuild its cache later from retained token IDs. That pause-and-release operation is called **preemption**. Rebuilding the lost K/V is called **recomputation**.

**Scope:** first-come-first-served priority, bounded token batches, chunked prompt processing, one-token decode appends, cancellation, and release-and-recompute preemption. The notebook executes plans explicitly. An automatic engine loop, sampling/EOS handling, CPU swapping and automatic GPU memory budgeting remain outside D4.

Run top to bottom on a **fresh Colab T4**. Two FP32 TinyLlama copies require about 8.8 GB of weights. The November 4, 2026 Colab run passed 304 tests with one two-GPU test skipped. Both timelines completed, including explicit preemption and prefix replay; the original evidence is archived in [D4_scheduler.json](../results/D4_scheduler.json). D3's separate failure/retry run is archived in [D3_block_manager.json](../results/D3_block_manager.json). Local checks remain static only. Save earlier results and successful notebook outputs before replacing their runtime.

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/guptasameer112/tinyserve.git"
REPO_DIR = "/content/tinyserve"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], check=True, capture_output=True, text=True).stdout)


## 1. Test the policy before using real weights

The next cell runs B8/C2/C3/D1/D2/D3 regression tests and the new D4 tests. A failed test stops the lesson. The C2 test requiring two GPUs may skip on one T4; CUDA model tests should run.

D4 tests exact scheduling timelines, new arrivals, token budgets, prompt chunking, cancellation and preemption. Twenty seeded random traces check that plans stay within budget, live block tables never overlap, logical progress never decreases, and finite admitted workloads finish. A seed makes the same random trace reproducible.

Policy-only tests simulate successful cache-length commits without computing attention. Separate small CPU/CUDA model tests actually run scheduled forwards, compare every new or replayed row with no-cache ours/Hugging Face, and inject a later-layer failure before retry. The real TinyLlama demos below provide another layer of evidence.

In [ ]:
import gc
from dataclasses import asdict
import torch
from tinyserve.basics import load_model, encode
from tinyserve.engine.scheduler import Scheduler
from tinyserve.models.llama import LlamaForCausalLM
from tinyserve.probe import compare_paged_batch
from tinyserve.envinfo import collect_env, save_result

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then rerun.")
DEVICE = "cuda"
torch.backends.cuda.matmul.allow_tf32 = False
torch.set_float32_matmul_precision("highest")
print(collect_env())
test_run = subprocess.run([sys.executable, "-m", "pytest", "tests/test_b8_model.py",
                           "tests/test_c2_generation.py", "tests/test_c3_cache.py",
                           "tests/test_d1_batching.py", "tests/test_d2_paged_cache.py",
                           "tests/test_d3_block_manager.py", "tests/test_d4_scheduler.py", "-v", "-ra"],
                          capture_output=True, text=True)
print(test_run.stdout)
print(test_run.stderr)
test_run.check_returncode()


## 2. A batch plan is a decision, not a forward

Our scheduler uses **first-come-first-served (FCFS)** priority: earlier arrivals get considered first. A **token budget** limits the number of new token rows in one forward. It does not limit the full cached history length or count requests.

With budget four, one request can process three prompt tokens and another can process one prompt token. That produces four rows. Both may attend to longer cached prefixes in later steps without counting those old prefix tokens as new rows.

Each planned item has a request ID, a starting position, an exclusive ending position, and a phase. For example, `[0,3)` means positions 0, 1 and 2, not position 3. The phases are:

- **Prefill:** process prompt tokens for the first time. A long prompt is split across steps if the budget is too small; this is chunked prefill.
- **Decode:** process one additional token after the prompt, using the existing prefix cache. Each request gets at most one decode token per step.
- **Recompute:** replay previously processed token IDs to rebuild K/V that was released during preemption. This restores cache state rather than adding new logical tokens.

`schedule()` chooses work and reserves blocks. The caller runs that work through the model. Only then can `complete_step()` acknowledge success and release finished requests. A request appears at most once in one plan, so finishing a replay and decoding its next token happen in separate steps.

D4 registers finite work using `add_request(request_id, prompt_tokens, decode_steps)`. Here, `decode_steps` counts extra token positions to process, **not sampled output tokens**. For this lesson, those extra token IDs come from fixed prompt slices. We are testing scheduling and cache correctness, not implementing generation or EOS stopping.

## 3. Load the references and record readable state

Load real TinyLlama in FP32, then copy its weights into our model. As in D2/D3, each planned forward is compared with separate no-cache forwards from our model and Hugging Face using the same full token prefix.

For every new or replayed logits row, the numerical tolerance is `abs(actual - expected) <= 1e-4 + 1e-5 * abs(expected)`. A separate assertion requires the exact same highest-scoring token ID on every row. The same readable paged-gather attention backend still runs; the scheduler changes which work enters it, not the attention equation.

The state snapshots below distinguish **logical progress** from **resident cache length**. Logical progress remembers how many tokens previously finished successfully. Resident length counts valid K/V entries currently present in this pool. They are usually equal, but preemption can remove the K/V without erasing logical history. A `null` resident length means the request currently owns no cache table.

The helper executes one explicit notebook step. It is not the automatic generation/serving engine planned for D5. Actual token IDs remain in `histories`, outside the scheduler and KV tensors, so they are available for replay.

In [ ]:
tokenizer, hf_model = load_model(device=DEVICE, dtype=torch.float32)
hf_model.set_attn_implementation("eager")
our_model = LlamaForCausalLM.from_huggingface(hf_model)
prompt_texts = ["The capital of France is", "The capital of Germany is",
                "A tiny language model can help people by"]
encoded_prompts = [encode(tokenizer, text, DEVICE) for text in prompt_texts]
assert all(tokens.numel() >= 5 for tokens in encoded_prompts)

def snapshot(scheduler):
    pool = scheduler.cache
    used = [block for table in pool.block_tables.values() for block in table]
    assert len(used) == len(set(used))
    assert set(used).isdisjoint(pool.free_blocks)
    assert set(used).union(pool.free_blocks) == set(range(pool.num_blocks))
    assert pool.ref_counts == tuple(int(block in used) for block in range(pool.num_blocks))
    return {"requests": {str(owner): {"status": state.status,
                                     "logical_progress": state.processed_tokens,
                                     "resident_cache_length": pool.lengths.get(owner)}
                         for owner, state in scheduler.requests.items()},
            "tables": {str(owner): list(table) for owner, table in pool.block_tables.items()},
            "free_blocks": list(pool.free_blocks)}

def execute_plan(label, scheduler, plan, histories):
    before = snapshot(scheduler)
    assert 1 <= plan.num_tokens <= scheduler.max_batch_tokens
    active = {item.request_id: histories[item.request_id][:item.end] for item in plan.work}
    comparison = compare_paged_batch(our_model, hf_model, scheduler.cache, active)
    assert len(comparison["requests"]) == len(plan.work)
    for item, row in zip(plan.work, comparison["requests"]):
        assert row["request_id"] == item.request_id
        assert row["start_position"] == item.start and row["committed_length"] == item.end
        assert row["new_tokens"] == item.end - item.start and row["all_row_argmax_equal"]
    scheduler.complete_step()
    result = {"label": label, "work": [asdict(item) for item in plan.work],
              "preempted": list(plan.preempted), "num_tokens": plan.num_tokens,
              "before_forward": before, "after_completion": snapshot(scheduler),
              "comparison": comparison}
    print(label, "work:", result["work"], "preempted:", result["preempted"])
    print("after completion:", result["after_completion"])
    return result

def summarize_scenario(name, scheduler, steps):
    assert scheduler.schedule() is None
    assert all(state.status == "finished" for state in scheduler.requests.values())
    assert not scheduler.cache.block_tables and not scheduler.cache.lengths
    assert scheduler.cache.num_free_blocks == scheduler.cache.num_blocks
    logical = sum(state.total_tokens for state in scheduler.requests.values())
    scheduled = sum(step["num_tokens"] for step in steps)
    recomputed = sum(item["end"] - item["start"] for step in steps for item in step["work"]
                     if item["phase"] == "recompute")
    assert scheduled == logical + recomputed
    return {"name": name, "num_blocks": scheduler.cache.num_blocks,
            "block_size": scheduler.cache.block_size, "max_batch_tokens": scheduler.max_batch_tokens,
            "pool_allocated_bytes": scheduler.cache.allocated_bytes,
            "logical_tokens": logical, "scheduled_tokens": scheduled, "recomputed_tokens": recomputed,
            "final_requests": {str(owner): asdict(state) for owner, state in scheduler.requests.items()},
            "final_pool": snapshot(scheduler), "steps": steps}


## 4. Start two requests with a four-token batch budget

Create five physical blocks, with two token slots per block. Request 0 has a three-token prompt plus two one-token appends: five logical tokens in total. Request 1 has a two-token prompt and no appends.

The first plan gives request 0 three prompt rows. Only one row of budget remains, so request 1 processes just its first prompt token. Its second prompt token waits for the next step. This is prompt chunking, not a lost token.

| Request | Planned positions | Rows | Logical progress before / after success |
|---|---|---|---|
| 0 | `[0,3)` | 3 | 0 / 3 |
| 1 | `[0,1)` | 1 | 0 / 1 |

Before running the model, we deliberately call `complete_step()` too early. It must raise because reserving blocks is not the same as processing tokens. We then execute the still-pending plan correctly. Returning the same pending plan makes retry possible; it does not rerun the model automatically.

Request 0 owns two blocks for its first three tokens; request 1 owns one block for its first token. Extra slots are capacity, not committed K/V. Both requests are still running after this step.

In [ ]:
continuous_pool = our_model.new_managed_paged_cache(5, 2)
continuous_pool.key.fill_(float("nan"))
continuous_pool.value.fill_(float("nan"))
continuous_scheduler = Scheduler(continuous_pool, max_batch_tokens=4)
continuous_histories = {0: encoded_prompts[0][:5], 1: encoded_prompts[1][:2], 2: encoded_prompts[2][:3]}
continuous_scheduler.add_request(0, prompt_tokens=3, decode_steps=2)
continuous_scheduler.add_request(1, prompt_tokens=2)
first_plan = continuous_scheduler.schedule()
before_early_completion = snapshot(continuous_scheduler)
try:
    continuous_scheduler.complete_step()
except RuntimeError as error:
    incomplete_completion_error = str(error)
    assert "not committed" in incomplete_completion_error
    print("expected early-completion error:", incomplete_completion_error)
else:
    raise AssertionError("a reservation must not count as a successful forward")
assert snapshot(continuous_scheduler) == before_early_completion
assert continuous_scheduler.schedule() is first_plan
assert [(item.request_id, item.start, item.end, item.phase) for item in first_plan.work] == [
    (0, 0, 3, "prefill"), (1, 0, 1, "prefill")]
continuous_steps = [execute_plan("step 0: two initial requests", continuous_scheduler, first_plan, continuous_histories)]


## 5. Let a new request join before the older one finishes

Request 2 arrives now, between completed forwards. It has a three-token prompt. We do **not** wait for request 0 to finish its two append steps before admitting the arrival.

The next two plans should be:

| Step | Work for request 0 | Work for request 1 | Work for request 2 | Total rows |
|---|---|---|---|---|
| 1 | Decode position 3 | Finish prompt at position 1 | Prefill positions 0,1 | 4 |
| 2 | Decode position 4 | Already finished | Finish prompt at position 2 | 2 |

This changing batch membership is **continuous batching**. Step 1 mixes decode and prefill in one forward. After that forward succeeds, request 1 reaches its complete workload and releases its blocks. Requests 0 and 2 finish after step 2. Finishing means the requested token-processing work is done; it is not an EOS decision in this fixed-input demo.

Notice that the final plan uses only two rows despite a budget of four. A budget is an upper bound, not an instruction to fabricate extra tokens. Decode is limited to one token per request per step.

In [ ]:
continuous_scheduler.add_request(2, prompt_tokens=3)
arrival_plan = continuous_scheduler.schedule()
assert [(item.request_id, item.start, item.end, item.phase) for item in arrival_plan.work] == [
    (0, 3, 4, "decode"), (1, 1, 2, "prefill"), (2, 0, 2, "prefill")]
continuous_steps.append(execute_plan("step 1: arrival joins mixed work", continuous_scheduler, arrival_plan, continuous_histories))
assert continuous_scheduler.requests[1].status == "finished" and 1 not in continuous_pool.block_tables
final_continuous_plan = continuous_scheduler.schedule()
assert [(item.request_id, item.start, item.end, item.phase) for item in final_continuous_plan.work] == [
    (0, 4, 5, "decode"), (2, 2, 3, "prefill")]
continuous_steps.append(execute_plan("step 2: remaining requests finish", continuous_scheduler, final_continuous_plan, continuous_histories))
continuous_result = summarize_scenario("continuous_batching", continuous_scheduler, continuous_steps)
assert continuous_result["logical_tokens"] == continuous_result["scheduled_tokens"] == 10
assert continuous_result["recomputed_tokens"] == 0


## 6. Use a smaller pool to force preemption

Now create a separate pool with only **three blocks of two slots**. Requests 0 and 1 each have a two-token prompt plus three one-token appends: five tokens per request. Each request fits alone, since five tokens need three blocks. Both complete histories cannot fit together in six physical slots.

We use a batch budget of three. Walk through this expected timeline before running the cell:

| Step | Request 0 work | Request 1 work | Why? |
|---|---|---|---|
| 0 | Prefill positions 0,1 | Prefill position 0 | Three-row budget is full |
| 1 | Decode position 2 | Prefill position 1 | All three blocks are now assigned |
| 2 | Decode position 3 | Wait | Request 1 needs a second block, but none is free |
| 3 | Decode position 4, then finish | Preempted | Request 0 needs its third block to finish |
| 4 | Already finished | Recompute positions 0,1 | Rebuild the lost prefix |
| 5 | Already finished | Decode position 2 | New work can resume |
| 6 | Already finished | Decode position 3 | Continue normally |
| 7 | Already finished | Decode position 4, then finish | All work is complete |

At step 3 the older request has priority. The scheduler releases request 1's block and gives the needed capacity to request 0. Request 1 is not cancelled: its logical progress remains two, and its token IDs remain in `pressure_histories`. Only its resident K/V is discarded. No data is swapped to CPU.

At step 4, request 1 gets blocks again and reprocesses its first two token IDs. Before replay, logical progress is two but resident cache length is zero. After replay, both are two. Logical progress does not become four: these are the same two tokens being recomputed, not two extra tokens added to the sequence.

The scheduler considers newer resident requests as eviction victims, newest first. It never evicts work already selected in that plan. A lower-priority request simply waits if only older requests hold the blocks it needs. Requests that cannot fit even alone are rejected when added, rather than waiting forever.

In [ ]:
pressure_pool = our_model.new_managed_paged_cache(3, 2)
pressure_pool.key.fill_(float("nan"))
pressure_pool.value.fill_(float("nan"))
pressure_scheduler = Scheduler(pressure_pool, max_batch_tokens=3)
pressure_histories = {0: encoded_prompts[0][:5], 1: encoded_prompts[1][:5]}
for request_id in pressure_histories:
    pressure_scheduler.add_request(request_id, prompt_tokens=2, decode_steps=3)
pressure_steps = []
for tick in range(16):
    pressure_plan = pressure_scheduler.schedule()
    if pressure_plan is None:
        break
    if pressure_plan.preempted:
        assert tick == 3 and pressure_plan.preempted == (1,)
        assert pressure_scheduler.requests[1].processed_tokens == 2
        assert pressure_scheduler.requests[1].status == "preempted" and 1 not in pressure_pool.lengths
    if tick == 4:
        assert [(item.request_id, item.start, item.end, item.phase) for item in pressure_plan.work] == [
            (1, 0, 2, "recompute")]
        assert pressure_pool.lengths[1] == 0 and pressure_scheduler.requests[1].processed_tokens == 2
    pressure_steps.append(execute_plan("pressure step %d" % tick, pressure_scheduler, pressure_plan, pressure_histories))
else:
    raise AssertionError("finite preemption demo did not finish within sixteen steps")
pressure_result = summarize_scenario("preemption_recompute", pressure_scheduler, pressure_steps)
assert len(pressure_steps) == 8
assert [owner for step in pressure_steps for owner in step["preempted"]] == [1]
assert pressure_result["logical_tokens"] == 10
assert pressure_result["scheduled_tokens"] == 12 and pressure_result["recomputed_tokens"] == 2
print("Ten logical token rows required twelve model rows because two prefix rows were recomputed.")


## 7. What this policy does not promise

Oldest-first priority is easy to understand, but it does not guarantee low latency or fair sharing for new arrivals. A long older prompt can consume the whole token budget. Preemption also has a cost: the second example processes twelve rows for ten logical tokens because it rebuilds a two-token prefix.

The pool size stays fixed. Releasing a request returns block IDs for reuse; it does not shrink the GPU tensors or securely erase their bytes. This lesson claims neither a speedup nor a GPU allocation reduction. The no-cache comparisons are correctness references, not benchmark timings.

The scheduler owns an initially empty managed cache. Do not separately reset, reserve or release its requests. It supports one pending plan at a time and is not thread-safe. Add requests or cancel them only between completed steps. Cancellation releases any blocks and makes a request terminal; preemption preserves it for later resumption.

If a model forward fails, do not call `complete_step()` as though it succeeded. The same pending plan remains available for an explicit retry. Its reservations and earlier preemptions remain in effect. D4's CPU/CUDA model tests check this contract; D3 explains how the cache protects committed prefixes during a failed forward.

## 8. Save complete evidence, then answer three questions

Before saving, the next cell requires both scenarios, all eleven completed forward records, the expected preemption, and the recomputation record. This guards against a result file that has a success flag but is missing an important stage. Each comparison includes the actual token IDs, positions, request IDs, physical slots and numerical differences.

1. **What makes batching continuous?** New requests can join between forwards while older requests continue; the original group does not have to finish together.
2. **What survives recomputation preemption?** Logical progress and retained token IDs survive. Resident K/V blocks are released, then rebuilt by replaying the same IDs.
3. **Why can a smaller pool create extra model work?** A request may lose a prefix cache and have to compute it again. The replay rows consume budget without adding new logical tokens.

Save `D4_scheduler.json` and the successful notebook with outputs before replacing the runtime. Numerical ranges below summarize correctness differences, not performance. **Next after validation: D5, the automatic engine loop and memory budgeting.**

In [ ]:
scenario_results = [continuous_result, pressure_result]
assert [result["name"] for result in scenario_results] == ["continuous_batching", "preemption_recompute"]
assert [len(result["steps"]) for result in scenario_results] == [3, 8]
assert any(item["phase"] == "recompute" for step in pressure_result["steps"] for item in step["work"])
assert pressure_result["final_requests"]["1"]["preemptions"] == 1
all_comparison_rows = [row for result in scenario_results for step in result["steps"]
                       for row in step["comparison"]["requests"]]
assert all(row["all_row_argmax_equal"] for row in all_comparison_rows)
no_cache_diffs = [row["paged_vs_no_cache_max_abs_diff"] for row in all_comparison_rows]
hf_diffs = [row["paged_vs_hf_max_abs_diff"] for row in all_comparison_rows]
saved_path = save_result("D4_scheduler", {
    "dtype": "float32", "backend": "pytorch_paged_gather", "policy": "fcfs_newest_victim_recompute",
    "fixed_token_histories": True, "incomplete_completion_error": incomplete_completion_error,
    "scenarios": scenario_results,
    "paged_vs_no_cache_diff_range": {"min": min(no_cache_diffs), "max": max(no_cache_diffs)},
    "paged_vs_hf_diff_range": {"min": min(hf_diffs), "max": max(hf_diffs)},
    "pytest": {"returncode": test_run.returncode, "stdout": test_run.stdout, "stderr": test_run.stderr},
})
print(saved_path)
print(saved_path.read_text())


In [ ]:
del continuous_scheduler, pressure_scheduler, continuous_pool, pressure_pool, our_model, hf_model
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory still held by this kernel: {torch.cuda.memory_allocated() / 2**20:.0f} MiB")
